# Silver Layer: Nation Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.nation` into `skyledger_airlines.silver.nation`.
Applies primary key validation (§3.1) and referential integrity checks (§3.4).
Invalid records are quarantined in `silver.nation_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `nation` → `passenger` and `nation` → `airline` (dual geographic hierarchy).
`region` → `nation` (region is the parent of nation). FK validated against `silver.region`.

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws
from pyspark.sql.window import Window

bronze_nation = spark.table("skyledger_airlines.bronze.nation")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast key columns from string to INT for proper numeric handling
nation_typed = bronze_nation.select(
    col("n_nationkey").cast("int").alias("n_nationkey"),
    col("n_name").cast("string").alias("n_name"),
    col("n_regionkey").cast("int").alias("n_regionkey"),
    col("n_comment").cast("string").alias("n_comment"),
)

bronze_count = nation_typed.count()
print(f"Bronze nation rows read: {bronze_count}")
display(nation_typed.limit(10))

In [0]:
# ── Step 2: Primary Key Validation ──
# Rule: n_nationkey must be NOT NULL and UNIQUE across all 25 rows.
# Failure codes: NULL_PK, DUPLICATE_PK

nation_flagged = nation_typed.withColumn(
    "is_null_pk",
    when(col("n_nationkey").isNull(), lit(True)).otherwise(lit(False)),
)

# Window-based duplicate detection
pk_window = Window.partitionBy("n_nationkey")
nation_flagged = nation_flagged.withColumn(
    "_pk_count", _count("*").over(pk_window)
).withColumn(
    "is_duplicate_pk",
    when((col("n_nationkey").isNotNull()) & (col("_pk_count") > 1), lit(True)).otherwise(lit(False)),
)

null_pk_count = nation_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = nation_flagged.filter(col("is_duplicate_pk")).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 3: Referential Integrity Validation ──
# Rule: n_regionkey must resolve to a valid r_regionkey in silver.region.
# Failure code: INVALID_FK_REGION
#
# Parent-before-child order: region is the parent of nation.
# Uses silver.region (already validated) as the reference.

valid_region_keys = (
    spark.table("skyledger_airlines.silver.region")
    .select(col("r_regionkey").cast("int").alias("r_regionkey"))
    .filter(col("r_regionkey").isNotNull())
    .distinct()
)

# Left join: rows with no match (or null n_regionkey) get r_regionkey = null → invalid FK
nation_flagged = nation_flagged.join(
    valid_region_keys,
    nation_flagged["n_regionkey"] == valid_region_keys["r_regionkey"],
    "left",
).withColumn(
    "is_invalid_fk_region",
    when(col("r_regionkey").isNull(), lit(True)).otherwise(lit(False)),
)

invalid_fk_count = nation_flagged.filter(col("is_invalid_fk_region")).count()
print(f"INVALID_FK_REGION failures: {invalid_fk_count}")

In [0]:
# ── Step 4: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
nation_flagged = nation_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
        when(col("is_invalid_fk_region"), lit("INVALID_FK_REGION")),
    ),
)

# Clean records: passed every check (rejection_reason is empty string)
silver_nation = nation_flagged.filter(col("rejection_reason") == "").select(
    "n_nationkey", "n_name", "n_regionkey", "n_comment"
)

# Quarantined records: failed one or more checks
quarantine_nation = nation_flagged.filter(col("rejection_reason") != "").select(
    "n_nationkey", "n_name", "n_regionkey", "n_comment", "rejection_reason"
)

silver_count = silver_nation.count()
quarantine_count = quarantine_nation.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_nation.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.nation"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_nation.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.nation_quarantine"))

print("✅ Tables written: silver.nation, silver.nation_quarantine")

In [0]:
# ── Step 5: Bronze-Silver-Quarantine Reconciliation (§3.5) ──
# Bronze row count must equal Silver + Quarantine for the nation table.

bronze_total = spark.table("skyledger_airlines.bronze.nation").count()
silver_total = spark.table("skyledger_airlines.silver.nation").count()
quarantine_total = spark.table("skyledger_airlines.silver.nation_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("✅ Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 6: Data Quality Report ──
# Documents every check executed, even when zero rows fail.

dq_report = spark.createDataFrame(
    [
        ("nation", "NULL_PK", "n_nationkey must not be null", null_pk_count, "Quarantine"),
        ("nation", "DUPLICATE_PK", "n_nationkey must be unique across all rows", duplicate_pk_count, "Quarantine"),
        ("nation", "INVALID_FK_REGION", "n_regionkey must reference a valid r_regionkey in silver.region", invalid_fk_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.nation_quarantine"))
else:
    print("\nNo quarantined records — all 25 nation rows passed every check.")

# Show final Silver table
print(f"\n── Silver Nation Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.nation"))